In [ ]:
import fastf1 as f1
import pandas as pd

In [ ]:
dict_tracks = {
    1: ["Bahrain", 5.412, 15],                    # [1](https://www.f1-circuits.com/)
    2: ["Saudi Arabia", 6.174, 27],               # Jeddah Corniche Circuit [1](https://www.f1-circuits.com/)
    3: ["Australia", 5.278, 14],                  # Albert Park [1](https://www.f1-circuits.com/)
    4: ["Emilia Romagna (Imola)", 4.909, 19],     # Imola circuit [2](https://en.wikipedia.org/wiki/Imola_Circuit)
    5: ["Miami", 5.412, 19],                      # Miami International Autodrome [1](https://www.f1-circuits.com/)
    6: ["Spain", 4.657, 14],                      # Barcelona-Catalunya [1](https://www.f1-circuits.com/)
    7: ["Monaco", 3.337, 19],                     # Circuit de Monaco [1](https://www.f1-circuits.com/)
    8: ["Azerbaijan", 6.003, 20],                 # Baku City Circuit [1](https://www.f1-circuits.com/)
    9: ["Canada", 4.361, 14],                     # Circuit Gilles Villeneuve [1](https://www.f1-circuits.com/)
    10: ["Great Britain", 5.891, 18],             # Silverstone [1](https://www.f1-circuits.com/)
    11: ["Austria", 4.318, 10],                   # Red Bull Ring [1](https://www.f1-circuits.com/)
    12: ["France", 5.842, 15],                    # Paul Ricard current layout [3](https://en.wikipedia.org/wiki/Circuit_Paul_Ricard)
    13: ["Hungary", 4.381, 14],                   # Hungaroring [1](https://www.f1-circuits.com/)
    14: ["Belgium", 7.004, 19],                   # Spa-Francorchamps [1](https://www.f1-circuits.com/)
    15: ["Netherlands", 4.259, 14],               # Zandvoort [1](https://www.f1-circuits.com/)
    16: ["Italy (Monza)", 5.793, 11],             # Monza [1](https://www.f1-circuits.com/)
    17: ["Singapore", 4.940, 19],                 # Marina Bay Street Circuit [1](https://www.f1-circuits.com/)
    18: ["Japan", 5.807, 18],                     # Suzuka [1](https://www.f1-circuits.com/)
    19: ["United States", 5.513, 20],             # Circuit of the Americas [1](https://www.f1-circuits.com/)
    20: ["Mexico", 4.304, 17],                    # Autodromo Hermanos Rodriguez [1](https://www.f1-circuits.com/)
    21: ["Brazil", 4.309, 15],                    # Interlagos [1](https://www.f1-circuits.com/)
    22: ["Abu Dhabi", 5.281, 16],                 # Yas Marina [1](https://www.f1-circuits.com/)
}

In [ ]:
Tyres_standardisation = {
    1 : ["C1","C2","C3"],
    2 : ["C2","C3","C4"],
    3 : ["C2","C3","C5"],
    4 : ["C2","C3","C4"],
    5 : ["C2","C3","C4"],
    6 : ["C1","C2","C3"],
    7 : ["C3","C4","C5"],
    8 : ["C3","C4","C5"],
    9 : ["C3","C4","C5"],
    10 : ["C1","C2","C3"],
    11 : ["C3","C4","C5"],
    12 : ["C2","C3","C4"],
    13 : ["C2","C3","C4"],
    14 : ["C2","C3","C4"],
    15 : ["C1","C2","C3"],
    16 : ["C2","C3","C4"],
    17 : ["C3","C4","C5"],
    18 : ["C1","C2","C3"],
    19 : ["C2","C3","C4"],
    20 : ["C2","C3","C4"],
    21 : ["C2","C3","C4"],
    22 : ["C3","C4","C5"]}

In [ ]:
skip_rounds = [4, 11, 21] # sprint races

for round_number in range(1,3):       ## 22 races in 2022
    if round_number not in skip_rounds:
        temp = f1.get_session(2022, round_number, "FP3")
        temp.load(telemetry=True)
        temp_name = temp.event.EventName

        laps = temp.laps
        laps = laps.reset_index(drop=True)

        weather_data = temp.laps.get_weather_data()
        weather_data = weather_data.reset_index(drop=True)
        laps_weather = pd.concat([laps, weather_data.loc[:, ~(weather_data.columns == 'Time')]], axis=1)

        ## circuit data

        laps_weather[["Circuit","Track_length","Number_of_corners"]] = [dict_tracks[round_number][0],dict_tracks[round_number][1],dict_tracks[round_number][2]]

        ## season tyre names remapping

        hard,medium,soft = Tyres_standardisation[round_number]
        tyre_mapping = {
            "HARD" : hard,
            "MEDIUM" : medium,
            "SOFT" : soft}

        laps_weather["Compound"] = laps_weather["Compound"].replace(tyre_mapping)

        ## Track status splitting


        laps_weather["state_str"] = laps_weather["TrackStatus"].astype(str)

        for d in range(1,8):
            laps_weather[f"state_{d}"] = laps_weather["state_str"].str.contains(str(d)).astype(int)

        laps_weather = laps_weather.drop(columns=["TrackStatus"])


        ## output
        
        laps_weather.to_csv(f"../Rounds/Round_{round_number}_{temp_name}_FP3.csv")

In [ ]:
for round_number in range(1,3):       ## 22 races in 2022
    temp = f1.get_session(2022, round_number, "FP2")
    temp.load(telemetry=True)
    temp_name = temp.event.EventName

    laps = temp.laps
    laps = laps.reset_index(drop=True)

    weather_data = temp.laps.get_weather_data()
    weather_data = weather_data.reset_index(drop=True)
    laps_weather = pd.concat([laps, weather_data.loc[:, ~(weather_data.columns == 'Time')]], axis=1)

    ## circuit data

    laps_weather[["Circuit","Track_length","Number_of_corners"]] = [dict_tracks[round_number][0],dict_tracks[round_number][1],dict_tracks[round_number][2]]

    ## season tyre names remapping

    hard,medium,soft = Tyres_standardisation[round_number]
    tyre_mapping = {
        "HARD" : hard,
        "MEDIUM" : medium,
        "SOFT" : soft}

    laps_weather["Compound"] = laps_weather["Compound"].replace(tyre_mapping)

     ## Track status splitting

    laps_weather["state_str"] = laps_weather["TrackStatus"].astype(str)

    for d in range(1,8):
        laps_weather[f"state_{d}"] = laps_weather["state_str"].str.contains(str(d)).astype(int)

    laps_weather = laps_weather.drop(columns=["TrackStatus"])

    ## output
    
    laps_weather.to_csv(f"../Rounds/Round_{round_number}_{temp_name}_FP2.csv")

In [ ]:
for round_number in range(1,3):       ## 22 races in 2022
    temp = f1.get_session(2022, round_number, "FP1")
    temp.load(telemetry=True)
    temp_name = temp.event.EventName

    laps = temp.laps
    laps = laps.reset_index(drop=True)

    weather_data = temp.laps.get_weather_data()
    weather_data = weather_data.reset_index(drop=True)
    laps_weather = pd.concat([laps, weather_data.loc[:, ~(weather_data.columns == 'Time')]], axis=1)

    ## circuit data

    laps_weather[["Circuit","Track_length","Number_of_corners"]] = [dict_tracks[round_number][0],dict_tracks[round_number][1],dict_tracks[round_number][2]]

    ## season tyre names remapping

    hard,medium,soft = Tyres_standardisation[round_number]
    tyre_mapping = {
        "HARD" : hard,
        "MEDIUM" : medium,
        "SOFT" : soft}

    laps_weather["Compound"] = laps_weather["Compound"].replace(tyre_mapping)

    ## Track status splitting

    laps_weather["state_str"] = laps_weather["TrackStatus"].astype(str)

    for d in range(1,8):
        laps_weather[f"state_{d}"] = laps_weather["state_str"].str.contains(str(d)).astype(int)

    laps_weather = laps_weather.drop(columns=["TrackStatus"])

    ## output
    
    laps_weather.to_csv(f"../Rounds/Round_{round_number}_{temp_name}_FP1.csv")

In [ ]:
for round_number in range(1,3):       ## 22 races in 2022
    temp = f1.get_session(2022, round_number, "R")
    temp.load(telemetry=True)
    temp_name = temp.event.EventName

    laps = temp.laps
    laps = laps.reset_index(drop=True)

    weather_data = temp.laps.get_weather_data()
    weather_data = weather_data.reset_index(drop=True)
    laps_weather = pd.concat([laps, weather_data.loc[:, ~(weather_data.columns == 'Time')]], axis=1)

    ## circuit data

    laps_weather[["Circuit","Track_length","Number_of_corners"]] = [dict_tracks[round_number][0],dict_tracks[round_number][1],dict_tracks[round_number][2]]

    ## season tyre names remapping

    hard,medium,soft = Tyres_standardisation[round_number]
    tyre_mapping = {
        "HARD" : hard,
        "MEDIUM" : medium,
        "SOFT" : soft}

    laps_weather["Compound"] = laps_weather["Compound"].replace(tyre_mapping)

    ## Track status splitting

    laps_weather["state_str"] = laps_weather["TrackStatus"].astype(str)
    
    for d in range(1,8):
        laps_weather[f"state_{d}"] = laps_weather["state_str"].str.contains(str(d)).astype(int)

    laps_weather = laps_weather.drop(columns=["TrackStatus"])
    ## output
    
    laps_weather.to_csv(f"../Rounds/Round_{round_number}_{temp_name}_Race.csv")


‘1’: Track clear (beginning of session or to indicate the end
    of another status)

‘2’: Yellow flag (sectors are unknown)

‘3’: ??? Never seen so far, does not exist?

‘4’: Safety Car

‘5’: Red Flag

‘6’: Virtual Safety Car deployed

‘7’: Virtual Safety Car ending (As indicated on the drivers steering wheel, on tv and so on; status ‘1’ will mark the actual end)